In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

url1 = pd.read_csv(r'D:\graduation thesis\DataSet\合并二手房测试2.csv', header=None)
url1.columns = ['t_price','park', 'light','p_lot','water','business','transport','l_tax','hall','room','area','orientation','furnishment','level','height','architecture','follower','span','gdp','region','green_rate','c_population','c_area','c_water_resources']

# 除去标签之外，共有13个特征，数据集的大小为178，
# 下面将数据集分为训练集和测试集
from sklearn.model_selection import train_test_split
print(type(url1))

x, y = url1.iloc[1:, 1:10].values, url1.iloc[1:, 0].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=0)
feat_labels = url1.columns[1:]

# n_estimators：森林中树的数量
# n_jobs  整数 可选（默认=1） 适合和预测并行运行的作业数，如果为-1，则将作业数设置为核心数
forest = RandomForestClassifier(n_estimators=10000, random_state=0, n_jobs=-1)
forest.fit(x_train, y_train.astype("float").astype("int"))
 

# 下面对训练好的随机森林，完成重要性评估

# feature_importances_  可以调取关于特征重要程度

importances = forest.feature_importances_

print("重要性：", importances)

x_columns = url1.columns[1:]

indices = np.argsort(importances)[::-1]

x_columns_indices = []

for f in range(x_train.shape[1]):

    # 对于最后需要逆序排序，我认为是做了类似决策树回溯的取值，从叶子收敛

    # 到根，根部重要程度高于叶子。

    print("%2d) %-*s %f" % (f + 1, 30, feat_labels[indices[f]], importances[indices[f]]))

    x_columns_indices.append(feat_labels[indices[f]])

 

print(x_columns_indices)

print(x_columns.shape[0])

print(x_columns)

print(np.arange(x_columns.shape[0]))

 

# 筛选变量（选择重要性比较高的变量）

threshold = 0.15

x_selected = x_train[:, importances > threshold]

# 可视化

import matplotlib.pyplot as plt

 

plt.figure(figsize=(10, 6))

plt.title("数据集中各个特征的重要程度", fontsize=18)

plt.ylabel("import level", fontsize=15, rotation=90)

plt.rcParams['font.sans-serif'] = ["SimHei"]

plt.rcParams['axes.unicode_minus'] = False

for i in range(x_columns.shape[0]):

    plt.bar(i, importances[indices[i]], color='orange', align='center')

    plt.xticks(np.arange(x_columns.shape[0]), x_columns_indices, rotation=90, fontsize=15)

plt.show()

d:\anaconda\lib\site-packages\IPython\core\interactiveshell.py:3331: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


<class 'pandas.core.frame.DataFrame'>


MemoryError: could not allocate 15876096 bytes

In [ ]:
# 可视化

import matplotlib.pyplot as plt

 

plt.figure(figsize=(10, 6))

plt.title("数据集中各个特征的重要程度", fontsize=18)

plt.ylabel("import level", fontsize=15, rotation=90)

plt.rcParams['font.sans-serif'] = ["SimHei"]

plt.rcParams['axes.unicode_minus'] = False

for i in range(x_columns.shape[0]):

    plt.bar(i, importances[indices[i]], color='orange', align='center')

    plt.xticks(np.arange(x_columns.shape[0]), x_columns_indices, rotation=90, fontsize=15)

plt.show()

In [3]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.svm import SVC
import matplotlib.pyplot as plt
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

url1 = pd.read_csv(r'D:\graduation thesis\DataSet\合并二手房测试2.csv', header=None)
url1.columns = ['t_price','park', 'light','p_lot','water','business','transport','l_tax','hall','room','area','orientation','furnishment','level','height','architecture','follower','span','gdp','region','green_rate','c_population','c_area','c_water_resources']

# 下面将数据集分为训练集和测试集
from sklearn.model_selection import train_test_split
print(type(url1))

X, y = url1.iloc[1:, 0].values, url1.iloc[1:, 1:].values

plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

# 不采用样本均衡SVC模型效果
svc_none_class_weight = SVC(kernel='linear').fit(X, y)

# 采用样本均衡参数class_weight
svc_with_class_weight = SVC(kernel='linear', class_weight={1: 10}).fit(X, y)

<class 'pandas.core.frame.DataFrame'>


IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed

In [ ]:
# -*- coding: utf-8 -*-
 
import cv2
import numpy as np
 
# 读取图片
imagePath = '/data/download/test1.jpg'
img = cv2.imread(imagePath)
 
# 转化成灰度图
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
 
# 利用Sobel边缘检测生成二值图
sobel = cv2.Sobel(gray, cv2.CV_8U, 1, 0, ksize=3)
# 二值化
ret, binary = cv2.threshold(sobel, 0, 255, cv2.THRESH_OTSU + cv2.THRESH_BINARY)
 
# 膨胀、腐蚀
element1 = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 9))
element2 = cv2.getStructuringElement(cv2.MORPH_RECT, (24, 6))
 
# 膨胀一次，让轮廓突出
dilation = cv2.dilate(binary, element2, iterations=1)
 
# 腐蚀一次，去掉细节
erosion = cv2.erode(dilation, element1, iterations=1)
 
# 再次膨胀，让轮廓明显一些
dilation2 = cv2.dilate(erosion, element2, iterations=2)
 
#  查找轮廓和筛选文字区域
region = []
contours, hierarchy = cv2.findContours(dilation2, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
for i in range(len(contours)):
    cnt = contours[i]
 
    # 计算轮廓面积，并筛选掉面积小的
    area = cv2.contourArea(cnt)
    if (area < 1000): continue # 找到最小的矩形 rect = cv2.minAreaRect(cnt) print ("rect is: ") print (rect) # box是四个点的坐标 box = cv2.boxPoints(rect) box = np.int0(box) # 计算高和宽 height = abs(box[0][1] - box[2][1]) width = abs(box[0][0] - box[2][0]) # 根据文字特征，筛选那些太细的矩形，留下扁的 if (height > width * 1.3):
        continue
 
    region.append(box)
 
# 绘制轮廓
for box in region:
    cv2.drawContours(img, [box], 0, (0, 255, 0), 2)
 
cv2.imshow('img', img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

meanX, stdX=10,0.3
n=1000 #这里的n就是蒙特卡洛模拟的随机数生成器
x=np.random.normal(meanX, stdX, n) #使用numpy内置的正态分布函数random.normal(),随机产生x1000次

y=x*10+x**2+5
np.shape(y) #用numpy的shape函数可以查看变量的大小,结果显示y为一个1000行的数组
meanY=np.mean(y)
stdY=np.std(y)

# 查看一下x和y随机产生了那些数值
fig,axes=plt.subplots(1,2,figsize=(8,4),constrained_layout=True)
axes[0].bar(np.arange(n),x)
axes[1].bar(np.arange(n),y)
axes[0].set_xlabel('n')
axes[0].set_ylabel('x')
axes[1].set_xlabel('n')
axes[1].set_ylabel('y')
plt.show()

#最后将x和y根据正态分布函数可视化
countX, binsX, ignoredX = plt.hist(x, 30, density=True)
plt.plot(binsX, 1/(stdX * np.sqrt(2 * np.pi)) *
               np.exp( - (binsX - meanX)**2 / (2 * stdX**2) ),
         linewidth=2, color='r')
plt.xlabel('x')
plt.show()

countY, binsY, ignoredY = plt.hist(y, 30, density=True)
plt.plot(binsY, 1/(stdY * np.sqrt(2 * np.pi)) *
               np.exp( - (binsY - meanY)**2 / (2 * stdY**2) ),
         linewidth=2, color='b')
plt.xlabel('y')
plt.show()